In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm
import gc

# CONFIG

In [ ]:
SEED = 42
NFOLDS = 5 #changed
MAX_LEN = 256//2
BATCH_SIZE = 16*2*2
EPOCHS = 10 #changed
EPOCHS2= 2# 2nd stage epochs
MODEL_PATH2 = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base" #.914
MODEL_PATH1='/kaggle/input/deberta-v3-small/transformers/default/1'
MODEL_PATH= '/kaggle/input/aug-models/deberta-v3-xsmall/' #.900
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Set seeds
import random
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# Load and preprocess data
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [ ]:
df = pd.read_csv(train_path)
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

In [ ]:
unlabelled_path='/kaggle/input/jigsaw-2m-reddit-unlabelled/reddit-removal-log.csv'
unlabelled_df = pd.read_csv(unlabelled_path)

In [ ]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [ ]:
test_df = pd.read_csv(test_path)

# Get augmented data from both df and test_df
augmented_train = add_data(df)
augmented_test = add_data(test_df)

# Combine original df with augmented data
augmented_texts =  df.text.tolist()+augmented_train[0] + augmented_test[0]
augmented_labels =  df.label.tolist()+augmented_train[1] + augmented_test[1]


# Create new augmented dataframe
augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before:{augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',  # Keep the original case of the first occurrence
    'label': 'mean'   # Take mean of labels
})
print('After:',augmented_df.shape)
augmented_df['rule']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

rule_map= {i:j for j,i in enumerate(augmented_df.rule.str.lower().unique())}
augmented_df['rule_id']= augmented_df.rule.str.lower().map(rule_map)

augmented_df.head()

In [ ]:
# Load tokenizer locally
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH2, use_fast = False)

In [ ]:
# print(df.head())
# print(df.columns.tolist())

In [ ]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels,rule_ids, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        return item

In [ ]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits = model(input_ids, mask)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
def validate(model, loader):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            rule_ids = batch["rule_ids"]  # Assuming this is already on CPU as integers
            
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    # Convert to numpy arrays
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    # Compute AUC per rule
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        # Only compute AUC if we have both positive and negative samples for this rule
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            # If only one class present, we can't compute AUC
            rule_aucs[rule_id] = np.nan
    
    # Compute average AUC across rules (excluding NaN values)
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    val_loss = total_loss / len(loader)
    # print(rule_aucs,'Rule_AUC')
    return avg_auc_per_rule, val_loss, preds,targets #XXX

In [ ]:
from transformers import get_linear_schedule_with_warmup

In [ ]:
#2 ideas, find similar ones to current +ves see if they are good, 2.find examples which are positive, sample 20 times more from their subreddits. 2.2 find where exclusive + examples are sampled from sample from that subreddits

In [ ]:
#Dealing with this -> 1. margin ranking loss with multiple pairs from same +ve and differnt-ve 2. unlabelled data working 3. llm data generation of positive points from rules and unlabelled data. 

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import semantic_search, dot_score
from tqdm import tqdm
import re

def clean_text(text):
  """Clean text for better embedding quality"""
  if pd.isna(text) or text == "":
        return ""
  # text = text.lower()
  # text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
  # text = re.sub(r'\S*@\S*\s?', '', text)
  # text = re.sub(r'\s+', ' ', text)
  # text = re.sub(r'[^\w\s.,!?;:\'"()-]', '', text)
  # text = text.strip()
  return text

def get_positives_from_augmented_df(augmented_df):
  """Extract positive examples grouped by rule from augmented_df"""
  positives_by_rule = {}

  for rule in augmented_df['rule'].unique():
      rule_data = augmented_df[augmented_df['rule'] == rule]
      # Get positive examples (label >= 0.5)
      positives = rule_data[rule_data['label'] >= 0.5]['body'].tolist()
      positives = [p for p in positives if pd.notna(p) and len(str(p).strip()) > 0]
      positives = list(set(positives))  # Remove duplicates

      positives_by_rule[rule] = positives
      print(f"Rule '{rule[:50]}...': {len(positives)} positive examples")

  return positives_by_rule

def find_similar_unlabelled_examples(positive_examples, unlabelled_candidates, 
                                embedding_model, sample_size, similarity_threshold=0.3):
  """
  Find unlabelled examples most similar to positive examples using semantic similarity
  
  Args:
      positive_examples: List of positive example texts
      unlabelled_candidates: DataFrame of unlabelled candidates  
      embedding_model: SentenceTransformer model
      sample_size: Number of samples to return
      similarity_threshold: Minimum similarity score to consider
  
  Returns:
      DataFrame of most similar unlabelled examples
  """
  if len(positive_examples) == 0 or len(unlabelled_candidates) == 0 or sample_size<10:
      return pd.DataFrame()

  # Limit candidates if too many (for performance)
  if len(unlabelled_candidates) > 10000:
      unlabelled_candidates = unlabelled_candidates.sample(n=10000, random_state=42)

  # Clean texts
  clean_positives = [clean_text(text) for text in positive_examples]
  candidate_bodies = unlabelled_candidates['body'].fillna('').astype(str).tolist()
  clean_candidates = [clean_text(text) for text in candidate_bodies]

  # Generate embeddings
  print(f"  Encoding {len(clean_positives)} positives and {len(clean_candidates)} candidates...")

  positive_embeddings = embedding_model.encode(
      sentences=clean_positives,
      batch_size=128*8,
      convert_to_tensor=True,
      device="cuda",
      normalize_embeddings=True,
      show_progress_bar=False
  )

  candidate_embeddings = embedding_model.encode(
      sentences=clean_candidates,
      batch_size=128*8,
      convert_to_tensor=True,
      device="cuda",
      normalize_embeddings=True,
      show_progress_bar=False
  )

  # Semantic search to find most similar candidates
  search_results = semantic_search(
      query_embeddings=positive_embeddings,
      corpus_embeddings=candidate_embeddings,
      top_k=min(sample_size * 3, len(clean_candidates)),  # Get more than needed for filtering
      score_function=dot_score
  )

  # Aggregate similarity scores across all positive examples
  candidate_scores = {}
  for pos_idx, pos_results in enumerate(search_results):
      for result in pos_results:
          candidate_idx = result['corpus_id']
          similarity = result['score']

          if similarity >= similarity_threshold:
              if candidate_idx not in candidate_scores:
                  candidate_scores[candidate_idx] = []
              candidate_scores[candidate_idx].append(similarity)

  # Calculate mean similarity for each candidate
  candidate_mean_scores = {
      idx: np.mean(scores) for idx, scores in candidate_scores.items()
  }

  # Sort by mean similarity and select top candidates
  top_candidates = sorted(candidate_mean_scores.items(),
                        key=lambda x: x[1], reverse=True)[:sample_size]

  if not top_candidates:
      # Fallback: return random sample if no similar candidates found
      print(f"  No similar candidates found (threshold={similarity_threshold}), falling back to random sampling")
      return unlabelled_candidates.sample(n=min(sample_size, len(unlabelled_candidates)), random_state=42)

  # Get the selected candidates
  selected_indices = [idx for idx, score in top_candidates]
  selected_candidates = unlabelled_candidates.iloc[selected_indices].copy()

  # Add similarity scores for debugging
  similarity_scores = [score for idx, score in top_candidates]
  selected_candidates['similarity_score'] = similarity_scores

  print(f"  Selected {len(selected_candidates)} similar candidates (avg similarity: {np.mean(similarity_scores):.3f})")

  return selected_candidates

def sample_unlabelled_data_semantic(multiplier=5, similarity_threshold=0.3, 
                               embedding_model_name="/kaggle/input/all-minilm-l12-v2/pytorch/l12-v2/1"):
  """
  Enhanced unlabelled data sampling using semantic similarity to positive examples
  """
  # Load embedding model
  print(f"Loading embedding model: {embedding_model_name}")
  embedding_model = SentenceTransformer(embedding_model_name, device="cuda")

  # Get positive examples by rule
  print("Extracting positive examples from augmented_df...")
  positives_by_rule = get_positives_from_augmented_df(augmented_df)

  # Get subreddit proportions (reuse existing function)
  proportions, test_size = get_subreddit_proportions()
  total_sample_size = test_size * multiplier

  # Calculate samples per rule based on augmented data rule distribution
  rule_counts = augmented_df["rule"].value_counts()
  rule_proportions = rule_counts / rule_counts.sum()

  # Prepare unlabelled data (reuse existing filtering logic)
  excluded_values = augmented_df['body'].values
  global unlabelled_df
  unlabelled_df = unlabelled_df.query('body not in @excluded_values')
  unlabelled_df.drop(unlabelled_df[(unlabelled_df['body'].str.len() > 2000)].index, inplace=True)
  unlabelled_df['body_lower'] = unlabelled_df.body.str.lower()
  unlabelled_df.drop_duplicates(subset=['body_lower'], keep='first', inplace=True, ignore_index=True)
  unlabelled_df.drop(columns=['body_lower'], inplace=True)

  sampled_data = []

  print("Performing semantic similarity-based sampling...")
  for rule, rule_prop in tqdm(rule_proportions.items(), desc="Processing rules"):
      rule_sample_size = int(total_sample_size * rule_prop)
      subreddit_props = proportions[rule]
      positive_examples = positives_by_rule.get(rule, [])

      if not positive_examples:
          print(f"  No positive examples for rule: {rule[:50]}...")
          continue

      rule_samples = []
      for subreddit, sub_prop in subreddit_props.items():
          subreddit_sample_size = int(rule_sample_size * sub_prop)

          if subreddit_sample_size == 0:
              continue

          # Get unlabelled candidates from this subreddit
          subreddit_data = unlabelled_df[
              unlabelled_df["subreddit"].str.lower().str.strip() == subreddit.lower().strip()
          ]

          if len(subreddit_data) == 0:
              continue

          print(f"\n  Rule: {rule[:30]}... | Subreddit: {subreddit} | Target: {subreddit_sample_size}")

          # Use semantic similarity instead of random sampling
          similar_samples = find_similar_unlabelled_examples(
              positive_examples=positive_examples,
              unlabelled_candidates=subreddit_data,
              embedding_model=embedding_model,
              sample_size=subreddit_sample_size,
              similarity_threshold=similarity_threshold
          )

          if len(similar_samples) > 0:
              similar_samples = similar_samples.copy()
              similar_samples["rule"] = rule
              rule_samples.append(similar_samples)

      if rule_samples:
          sampled_data.append(pd.concat(rule_samples, axis=0))

  # Combine all samples and shuffle
  if sampled_data:
      final_sample = pd.concat(sampled_data, axis=0).reset_index(drop=True)
      final_sample = final_sample.sample(frac=1, random_state=42).reset_index(drop=True)

      print(f"\n=== Sampling Results ===")
      print(f"Total samples: {len(final_sample)}")
      if 'similarity_score' in final_sample.columns:
          print(f"Average similarity: {final_sample['similarity_score'].mean():.3f}")
          print(f"Similarity range: {final_sample['similarity_score'].min():.3f} - {final_sample['similarity_score'].max():.3f}")

      return final_sample
  else:
      print("No samples found! Falling back to original random sampling...")
      return sample_unlabelled_data(multiplier)

In [ ]:
def get_subreddit_proportions():
    """Get subreddit proportions for each rule from test data"""
    prop_df = pd.concat((pd.read_csv(test_path),pd.read_csv(train_path)))
    
    proportions = {}
    for rule in prop_df["rule"].unique():
        rule_data = prop_df[prop_df["rule"] == rule]
        subreddit_counts = rule_data["subreddit"].value_counts()
        subreddit_props = subreddit_counts / subreddit_counts.sum()
        proportions[rule] = subreddit_props.to_dict()
    
    return proportions, len(prop_df)
    
def sample_unlabelled_data(multiplier=5):
    """Sample unlabelled data maintaining test data proportions"""
    proportions, test_size = get_subreddit_proportions()
    
    total_sample_size = test_size * multiplier
    sampled_data = []
    
    # Calculate samples per rule based on test data rule distribution
    rule_counts = augmented_df["rule"].value_counts()
    rule_proportions = rule_counts / rule_counts.sum()

    # unlabelled_df= pd.read_csv(unlabelled_path)
    excluded_values = augmented_df['body'].values# + train_df[['body', 'positive_example_1', 'positive_example_2', 'negative_example_1', 'negative_example_2']].values.ravel()
    global unlabelled_df
    unlabelled_df = unlabelled_df.query('body not in @excluded_values')
    
    unlabelled_df.drop(unlabelled_df[(unlabelled_df['body'].str.len() > 2000)].index, inplace=True)    #filter out body present in test,examples
    # print(unlabelled_df.shape)
    unlabelled_df['body_lower']=unlabelled_df.body.str.lower()
    unlabelled_df.drop_duplicates(subset=['body_lower'],keep='first',inplace=True,ignore_index=True)
    unlabelled_df.drop(columns=['body_lower'],inplace=True)
    # print(unlabelled_df.shape,'Only Unique')
    
    import tqdm
    for rule, rule_prop in tqdm.tqdm(rule_proportions.items()):
        rule_sample_size = int(total_sample_size * rule_prop)
        subreddit_props = proportions[rule]
        
        rule_samples = []
        for subreddit, sub_prop in subreddit_props.items():
            subreddit_sample_size = int(rule_sample_size * sub_prop)
            
            # Sample from unlabelled data for this subreddit
            subreddit_data = unlabelled_df[unlabelled_df["subreddit"].str.lower().str.strip() == subreddit.lower().strip()]
            if len(subreddit_data) ==0 or subreddit_sample_size==0: 
                # Fallback: sample from any data for this rule
                continue
                # subreddit_data = unlabelled_df[unlabelled_df["rule"] == rule]

            if len(subreddit_data) >= subreddit_sample_size:
                sampled = subreddit_data.sample(n=subreddit_sample_size, random_state=42)
            else:
                # If not enough data, sample with replacement
                sampled = subreddit_data.copy()
            
            sampled = sampled.copy()
            sampled["rule"] = rule
            rule_samples.append(sampled)
        
        if rule_samples:
            sampled_data.append(pd.concat(rule_samples, axis=0))
    
    # Combine all samples and shuffle
    final_sample = pd.concat(sampled_data, axis=0).reset_index(drop=True)
    final_sample = final_sample.sample(frac=1, random_state=42).reset_index(drop=True)
    
    return final_sample
# mlm_data= sample_unlabelled_data(1)
# mlm_data.head()

In [ ]:
mlm_data= sample_unlabelled_data_semantic(1)#10
mlm_data.head()

In [ ]:
mlm_data["text"] = mlm_data["rule"] + " [SEP] " + mlm_data["body"]#changed epochs, folds, semantic sample data, nots, sample size

In [ ]:
import math

In [ ]:
mlm_texts = mlm_data.text.tolist() #prepare_prepare_mlm_data(df_train, df_test)(df, df_test_full)

from sklearn.model_selection import train_test_split

mlm_train_texts, mlm_val_texts = train_test_split(
    mlm_texts, test_size=0.1, random_state=SEED
)

In [ ]:
# Prepare data for pseudo labeling
pseudo_data = mlm_data.copy()
pseudo_data["text"] = pseudo_data["rule"] + " [SEP] " + pseudo_data["body"]
pseudo_data["rule_id"] = pseudo_data["rule"].str.lower().map(rule_map)
# pseudo_data['label']= pseudo_data['mean_prediction'].round(0).astype(np.int8)

In [ ]:
# balanced_pseudo_data_useful= pd.concat([pseudo_data_useful.query('label==0').sample(frac=.4,replace=False),pseudo_data_useful.query('label==1')],axis=0)
# balanced_pseudo_data_useful.label.value_counts()

In [ ]:
# pseudo_data_95= pseudo_data[(pseudo_data['mean_prediction']>.95) | (pseudo_data['mean_prediction']<.05)]
# pseudo_data_9= pseudo_data[(pseudo_data['mean_prediction']>.9) | (pseudo_data['mean_prediction']<.1)]

In [ ]:
# len(pseudo_data_95),len(augmented_df)

In [ ]:
# balanced_pseudo_data_9= pd.concat([pseudo_data_9.query('label==0').sample(frac=.4,replace=False),pseudo_data_9.query('label==1')],axis=0)
# balanced_pseudo_data_9.label.value_counts()

In [ ]:
import gc;gc.collect()
torch.cuda.memory.empty_cache()

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # STAGE 1: Train single model for pseudo labeling (CHANGED: single model instead of 5-fold CV)
    print("=== STAGE 1: Training single model for pseudo labeling ===")

    # Create 90/10 split instead of 5-fold CV
    train_data, val_data = train_test_split(
        augmented_df,
        test_size=0.1,
        stratify=augmented_df["rule"],
        random_state=SEED
    )

    print(f"Train size: {len(train_data)} (90%), Val size: {len(val_data)} (10%)")
    torch.cuda.empty_cache()
    gc.collect()

    # Create datasets
    train_ds = JigsawDataset(
        train_data['text'].tolist(),
        train_data['label'].tolist(),
        train_data['rule_id'].tolist(),
        tokenizer, MAX_LEN
    )

    val_ds = JigsawDataset(
        val_data['text'].tolist(),
        val_data['label'].tolist(),
        val_data['rule_id'].tolist(),
        tokenizer, MAX_LEN
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    model = JigsawModel(MODEL_PATH2).to(DEVICE)
    for name, param in model.named_parameters():
        if name.startswith('base.embedding'):
            param.requires_grad = False
    model = nn.DataParallel(model)
    print('Trainable Params:',sum(i.numel() for i in model.parameters() if i.requires_grad))

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, eps=1e-6)
    total_steps = EPOCHS * len(train_loader)
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    best_auc = 0
    for epoch in range(EPOCHS):
        print(f"Epoch {epoch+1}/{EPOCHS}")
        loss = train_one_epoch(model, train_loader, optimizer, scheduler)
        val_auc, val_loss, val_preds,_ = validate(model, val_loader)

        print(f"Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            model_to_save = model.module if hasattr(model, 'module') else model
            torch.save(model_to_save.state_dict(), "model_stage1_best.bin")

    print(f"Stage 1 Best AUC: {best_auc:.4f}")
    del optimizer, scheduler, train_ds, val_ds, train_loader, val_loader

    # INFERENCE: Use single model for pseudo labeling
    model.eval()
    test_ds = JigsawDataset(pseudo_data['text'].tolist(), [0]*len(pseudo_data),[0]*len(pseudo_data), tokenizer, MAX_LEN)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2)

    fold_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Generating pseudo labels"):
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            logits = model(ids, mask)
            fold_preds.extend(torch.sigmoid(logits).cpu().numpy())

    pseudo_data['mean_prediction'] = fold_preds
    del model
    torch.cuda.empty_cache()

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # STAGE 2: Train single model with augmented + pseudo data (CHANGED: single model instead of 5-fold CV)
    print("=== STAGE 2: Training single model with pseudo labels ===")

    # Combine augmented data + pseudo data
    combined_texts = augmented_df['text'].tolist() + pseudo_data['text'].tolist()
    combined_labels = augmented_df['label'].tolist() + pseudo_data['mean_prediction'].tolist()
    combined_rule_ids = augmented_df['rule_id'].tolist() + pseudo_data['rule_id'].tolist()
    combined_rules = augmented_df['rule'].tolist() + pseudo_data['rule'].tolist()

    combined_df = pd.DataFrame({
        'text': combined_texts,
        'label': combined_labels,
        'rule_id': combined_rule_ids,
        'rule': combined_rules
    })

    print(f"Combined data size: {len(combined_df)} (augmented: {len(augmented_df)}, pseudo: {len(pseudo_data)})")

    # Create 90/10 split on combined data
    train_combined, val_combined = train_test_split(
        combined_df,
        test_size=0.1,
        stratify=combined_df["rule"],
        random_state=SEED
    )

    print(f"Combined train size: {len(train_combined)} (90%), Combined val size: {len(val_combined)} (10%)")
    torch.cuda.empty_cache()
    gc.collect()

    # Create datasets
    train_ds = JigsawDataset(
        train_combined['text'].tolist(),
        train_combined['label'].tolist(),
        train_combined['rule_id'].tolist(),
        tokenizer, MAX_LEN
    )

    val_ds = JigsawDataset(
        val_combined['text'].tolist(),
        val_combined['label'].tolist(),
        val_combined['rule_id'].tolist(),
        tokenizer, MAX_LEN
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    model = JigsawModel(MODEL_PATH).to(DEVICE)
    for name, param in model.named_parameters():
        if name.startswith('base.embedding'):
            param.requires_grad = False
    model = nn.DataParallel(model)
    print('Trainable Params:', sum(i.numel() for i in model.parameters() if i.requires_grad))

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, eps=1e-6)
    total_steps = EPOCHS2 * len(train_loader)
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    best_auc = 0
    for epoch in range(EPOCHS2):
        print(f"Epoch {epoch+1}/{EPOCHS2}")
        loss = train_one_epoch(model, train_loader, optimizer, scheduler)
        val_auc, val_loss, val_preds, _ = validate(model, val_loader)

        print(f"Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            model_to_save = model.module if hasattr(model, 'module') else model
            torch.save(model_to_save.state_dict(), "model_stage2_best.bin")

    print(f"Stage 2 Best AUC: {best_auc:.4f}")
    del model, optimizer, scheduler, train_ds, val_ds, train_loader, val_loader

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Generate final predictions using single Stage 2 model (CHANGED: single model instead of ensemble)
    print("=== Generating final predictions ===")

    sample = pd.read_csv(sample_sub_path)
    df_test = pd.read_csv(test_path)
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]

    # Load best Stage 2 model
    model = JigsawModel(MODEL_PATH).to(DEVICE)
    model.load_state_dict(torch.load("model_stage2_best.bin", map_location=DEVICE))
    model = nn.DataParallel(model)
    model.eval()

    test_ds = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test), [0]*len(df_test), tokenizer, MAX_LEN)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2)

    test_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Generating final predictions"):
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            logits = model(ids, mask)
            test_preds.extend(torch.sigmoid(logits).cpu().numpy())

    del model
    torch.cuda.empty_cache()

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Use single model predictions (no ensemble averaging needed)
    sample["rule_violation"] = test_preds
    sample.to_csv("submission.csv", index=False)
    print("✅ Submission saved as submission.csv")
else:
    !touch submission.csv
    
!head -n 4 submission.csv